# Thống kê cấu trúc tài liệu

Chương này trình bày các chỉ số thống kê về cây phân cấp tài liệu, mức độ làm sạch công thức toán và kết quả hợp nhất trích dẫn BibTeX.

Bảng và biểu đồ trong chương này sinh trực tiếp khi build từ tệp `report_book/data/m2_summary.json`. Các nhận định định lượng dùng ký hiệu `[...]` cho đến khi quá trình chạy bộ phân tích cú pháp trên toàn bộ dữ liệu hoàn tất.

## 1. Tổng quan số lượng phần tử

Bảng dưới đây tổng hợp kết quả bóc tách cấu trúc tài liệu LaTeX, xác định đường biên dịch và dựng cây phân cấp trên mọi phiên bản đã thu thập từ Giai đoạn 1.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

with open("../data/m2_summary.json", "r", encoding="utf-8") as f:
    m2_summary = json.load(f)

p_overview = m2_summary["parsing_overview"]

overview_df = pd.DataFrame([
    {"Chỉ số": "Tổng số bài báo xử lý", "Giá trị": f"{p_overview['total_publications_processed']:,}"},
    {"Chỉ số": "Tổng số phiên bản xử lý", "Giá trị": f"{p_overview['total_versions_processed']:,}"},
    {"Chỉ số": "Số phiên bản bóc tách cú pháp thành công", "Giá trị": f"{p_overview['successful_parsed_versions']:,}"},
    {"Chỉ số": "Số phiên bản dạng bản ghi rỗng", "Giá trị": f"{p_overview['empty_records_versions']:,}"},
    {"Chỉ số": "Tỉ lệ phân tích cú pháp thành công", "Giá trị": f"{p_overview['parsing_success_rate_percent']:.2f}%"}
])
overview_df.style.hide(axis="index").set_caption("Bảng 2.1: Tổng hợp kết quả phân tích cú pháp Giai đoạn 2")


### Nhận xét về tỉ lệ bóc tách cú pháp

- Tỉ lệ phân tích cú pháp thành công trên các phiên bản có mã TeX hợp lệ đạt `[...]%`. Thuật toán phát hiện tệp gốc chứa lệnh `\documentclass` và lần theo chuỗi `\input`, `\include` đã xử lý được cấu trúc tệp của các bài báo.
- Toàn bộ `[...]` phiên bản dạng PDF hoặc Word được ghi nhận dưới dạng bản ghi rỗng với 0 phần tử để giữ đúng mẫu số của tập dữ liệu.


## 2. Kết quả khử trùng lặp

Khử trùng lặp qua các phiên bản giúp giảm lượng dữ liệu thừa trên cả hai bảng `elements` và `bibtex`.


In [ ]:
dedup = m2_summary["deduplication_metrics"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), dpi=120)

el_cats = ["Trước khử trùng lặp", "Sau khử trùng lặp\n(Bảng elements)"]
el_vals = [dedup["elements_before_dedup"] / 1e6, dedup["elements_after_dedup"] / 1e6]
ax1.bar(el_cats, el_vals, color=["#ff7f00", "#377eb8"], width=0.45, edgecolor="black", linewidth=0.5)
ax1.set_ylabel("Triệu phần tử", fontsize=10)
ax1.set_title("Hình 2.1a: Số lượng phần tử nội dung", fontsize=11)
for i, v in enumerate(el_vals):
    ax1.text(i, v + 0.1, f"{v:.2f} M", ha="center", va="bottom", fontsize=10)
ax1.text(0.5, max(el_vals)*0.7, f"Giảm: {dedup['element_reduction_percent']:.1f}%", fontsize=11, ha="center",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="#fff2cc", edgecolor="#d6b656"))
ax1.set_ylim(0, max(el_vals) * 1.25)

bib_cats = ["Trước khử trùng lặp", "Sau khử trùng lặp\n(Bảng bibtex)"]
bib_vals = [dedup["bibtex_entries_before_dedup"] / 1e3, dedup["bibtex_entries_after_dedup"] / 1e3]
ax2.bar(bib_cats, bib_vals, color=["#e41a1c", "#4daf4a"], width=0.45, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Nghìn mục trích dẫn", fontsize=10)
ax2.set_title("Hình 2.1b: Số lượng mục trích dẫn", fontsize=11)
for i, v in enumerate(bib_vals):
    ax2.text(i, v + 15, f"{v:.1f} K", ha="center", va="bottom", fontsize=10)
ax2.text(0.5, max(bib_vals)*0.7, f"Giảm: {dedup['bibtex_reduction_percent']:.1f}%", fontsize=11, ha="center",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="#d5e8d4", edgecolor="#82b366"))
ax2.set_ylim(0, max(bib_vals) * 1.25)

plt.tight_layout()
plt.show()


### Nhận xét về mức độ cắt giảm dữ liệu

- Sau khi so khớp nội dung sau bước dọn dẹp, số lượng phần tử trong bảng `elements` giảm được `[...]%`. Số đo này cho thấy giữa các lần cập nhật của một bài báo, phần lớn các câu và công thức toán học không bị thay đổi.
- Số lượng mục trích dẫn trong bảng `bibtex` giảm `[...]%` sau khi gộp các khóa trích dẫn và hợp nhất các trường thông tin. Các lệnh `\cite` trong thân bài được đổi tên tương ứng để trỏ đúng khóa đã chọn.


## 3. Phân bố cấp bậc và dạng nút lá

Cây phân cấp của các bài báo có độ sâu trung bình đạt `[...]` tầng từ gốc tài liệu đến các nút lá. Đồ thị dưới đây thể hiện tỉ lệ của các dạng nút lá trên cây phân cấp.


In [ ]:
kinds = m2_summary["hierarchy_distribution"]["element_kinds"]

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
labels = ["Câu văn xuôi", "Khối công thức toán", "Hình ảnh", "Bảng biểu", "Mục liệt kê"]
vals = [kinds["sentence"], kinds["block_formula"], kinds["figure"], kinds["table"], kinds["item"]]
colors = ["#386cb0", "#fdb462", "#7fc97f", "#beaed4", "#fdc086"]

wedges, texts, autotexts = ax.pie(vals, labels=labels, autopct="%1.1f%%", startangle=140, colors=colors, explode=(0.04, 0.04, 0, 0, 0), textprops={"fontsize": 9})
ax.set_title("Hình 2.2: Tỉ lệ phân bố các dạng nút lá trên cây phân cấp", fontsize=12, pad=12)
plt.tight_layout()
plt.show()


### Nhận xét về cơ cấu nút lá

- Các câu văn xuôi chiếm `[...]%`, là phần nội dung chính mang ngữ cảnh và chứa các câu lệnh `\cite`.
- Các khối công thức toán chiếm `[...]%`, thể hiện đặc thù của các bài báo thuộc ngành máy tính và toán học trên arXiv.
- Các nút dạng hình ảnh và bảng biểu chiếm `[...]%`. Do giữ nguyên các lệnh `figure` và `\includegraphics`, cấu trúc chú thích và vị trí của hình ảnh được bảo tồn trên cây.


## 4. Các trường hợp ngoại lệ

Trong quá trình xử lý mã TeX từ nhiều tác giả, bộ phân tích cú pháp đã gặp một số dạng bất thường về cú pháp.

In [ ]:
errs = m2_summary["parse_errors_analysis"]

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
err_labels = [
    "Lệnh tự định nghĩa lồng nhau",
    "Lệch dấu ngoặc hoặc dấu toán",
    "Định dạng bibitem không chuẩn",
    "Ký tự bảng mã đặc thù"
]
err_vals = [errs["unconventional_macros"], errs["unbalanced_delimiters"], errs["non_standard_bib_style"], errs["encoding_and_foreign_chars"]]

bars = ax.barh(err_labels, err_vals, color="#bc80bd", edgecolor="black", linewidth=0.5)
ax.set_xlabel("Tỉ lệ trong tổng số trường hợp cảnh báo (%)", fontsize=10)
ax.set_title("Hình 2.3: Phân loại các nguyên nhân lỗi cú pháp LaTeX", fontsize=12, pad=12)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.8, bar.get_y() + bar.get_height()/2.0, f"{w:.1f}%", ha="left", va="center", fontsize=9)
ax.set_xlim(0, max(err_vals) * 1.2)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


Phân tích sơ bộ từ mẫu thử nghiệm cho thấy:

- Lệnh tự định nghĩa chiếm `[...]%`: Một số tác giả dùng lệnh `\def` để viết tắt công thức. Bộ phân tích cú pháp thay thế các macro cơ bản trước khi phân tích để đưa về dạng chuẩn.
- Lệch dấu ngoặc chiếm `[...]%`: Một số bài báo thiếu dấu ngoặc nhọn đóng hoặc dùng lẫn giữa `$ ... $` và `\begin{math}`. Bộ phân tích cú pháp dùng cơ chế kiểm tra dấu theo dòng để tránh lỗi dừng chương trình.